# Lab5C - Saving and Loading Models

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
cd "./gdrive/MyDrive/UCCD3074_Labs/UCCD3074_Lab5"

In [ ]:
# imports
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader

import torch.optim as optim
import os

from cifar10 import CIFAR10

In [ ]:
if not os.path.exists("./models"):
    os.mkdir("models")

## 1. Introduction

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, stride=1, padding=1)   
        self.bn1 = nn.BatchNorm2d(8)
        
        self.conv2 = nn.Conv2d(8, 16, 3)  
        self.bn2   = nn.BatchNorm2d(16)
        
        self.fc1 = nn.Linear(16*30*30, 256) 
        self.fc2 = nn.Linear(256, 10)
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        
        x = x.view(x.size(0), -1) # flat
        x = self.fc1(x)
        x = self.fc2(x)
        
        return x

In [ ]:
model = Net()

---

### 1.1 Saving & Loading Model Parameters Only

---
### 1.2 Saving the Entire Model

---
### 1.3 Saving the Model Parameters and Optimizer State

---

## 2. Example

#### Load the dataset

In [ ]:
# transform the model
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# dataset
trainset = CIFAR10(train=True,  transform=transform, num_samples=10000)
validset  = CIFAR10(train=False,  transform=transform, num_samples=2000)

# dataloader]
trainloader = DataLoader(trainset, batch_size=32, shuffle=True, num_workers=4)
validloader  = DataLoader(validset, batch_size=128, shuffle=True, num_workers=4)

#### Define training function

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def train(model, optimizer, start_epoch=0, max_epochs=10):
    
    # compute loss 3 times in each epoch
    loss_iterations = int(np.ceil(len(trainloader)/3))
    
    # transfer model to GPU
    model = model.to(device)
    
    # set the optimizer. Use SGD with momentum
    
    # set to training mode
    model.train()
        
    # train the network
    for e in range(start_epoch, max_epochs):    

        running_loss = 0
        running_count = 0

        for i, (inputs, labels) in enumerate(trainloader):

            # Clear all the gradient to 0
            optimizer.zero_grad()

            # transfer data to GPU
            inputs = inputs.to(device)
            labels = labels.to(device)

            # forward propagation to get h
            outs = model(inputs)

            # compute loss 
            loss = F.cross_entropy(outs, labels)

            # backpropagation to get gradients of all parameters
            loss.backward()

            # update parameters
            optimizer.step()

            # get the loss
            running_loss += loss.item()
            running_count += 1

             # display the averaged loss value 
            if i % loss_iterations == loss_iterations-1 or i == len(trainloader) - 1:    
                # compute training loss
                train_loss = running_loss / running_count
                running_loss = 0. 
                running_count = 0.
               
                print(f'[Epoch {e+1:2d} Iter {i+1:5d}/{len(trainloader)}]: train_loss = {train_loss:.4f}')       
        
        # save the model 
        ...

#### Train model for 2 epochs


In [ ]:
lr=0.01; momentum=0.9

model = Net()
optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)

train(model, optimizer, max_epochs=2)

#### Resume training

Resume training and train for another 2 epochs. To do that, we get the load the *previous* model's and optimizer's `state_dict`, the last epoch and training loss value.

In [ ]:
# define a new model
...

# define a new optimizer
...

# load the checkpoint file
...

# resume training
...